# GPU Memory

Prepare experiments that distinguish allocation, transfer, and access behavior in GPU workloads.

## Objectives

Reason about GPU-visible memory, access patterns, transfer costs, and measurement boundaries.

## Background

GPU performance depends on memory bandwidth, locality, coalescing, allocation behavior, and the platform's memory architecture.

## Prediction

PyTorch CUDA memory statistics should expose at least three distinct views of memory:

1. **allocated memory**: storage currently associated with live CUDA tensors;
2. **reserved memory**: memory held by PyTorch's caching allocator, including storage that is not currently assigned to a live tensor;
3. **CUDA-visible free memory**: memory the CUDA runtime reports as available to this process.

Allocating a large CUDA tensor should increase both allocated and reserved memory.

Deleting the tensor and running Python garbage collection should reduce allocated memory because the tensor storage is no longer live. Reserved memory may remain high because PyTorch normally keeps released blocks in its caching allocator for reuse.

Calling `torch.cuda.empty_cache()` should release unused cached blocks back to CUDA. It should reduce reserved memory and increase CUDA-visible free memory, but it should not release storage belonging to any tensor that is still live.

Creating a tensor with `torch.empty` establishes CUDA-visible storage but does not initialize its contents. Therefore this experiment measures allocator accounting, not the amount of physical LPDDR5X traffic or the residency of every page in unified memory.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [2]:
from pprint import pprint

import torch

from common.cuda import detect_cuda


cuda_info = detect_cuda()
pprint(cuda_info)

if not cuda_info.torch_installed:
    raise RuntimeError(
        "PyTorch is not installed in this environment. "
        "GPU memory experiments cannot continue."
    )

if not cuda_info.available:
    raise RuntimeError(
        "PyTorch is installed, but CUDA is not available. "
        f"Detection error: {cuda_info.error!r}"
    )

device_index = torch.cuda.current_device()
device = torch.device(f"cuda:{device_index}")
device_properties = torch.cuda.get_device_properties(device_index)

device_memory_info = {
    "device_index": device_index,
    "device_name": torch.cuda.get_device_name(device_index),
    "compute_capability": (
        device_properties.major,
        device_properties.minor,
    ),
    "total_cuda_memory_bytes": device_properties.total_memory,
    "total_cuda_memory_gib": device_properties.total_memory / 1024**3,
}

pprint(device_memory_info)

CudaInfo(torch_installed=True,
         available=True,
         device_count=1,
         device_names=('NVIDIA GB10',),
         torch_version='2.13.0+cu130',
         cuda_version='13.0',
         error=None)
{'compute_capability': (12, 1),
 'device_index': 0,
 'device_name': 'NVIDIA GB10',
 'total_cuda_memory_bytes': 130663002112,
 'total_cuda_memory_gib': 121.68940353393555}


### PyTorch CUDA allocator accounting

PyTorch uses a caching allocator for CUDA tensor storage. Releasing a tensor does not necessarily return its storage immediately to the CUDA runtime. The allocator may retain the block so that a later tensor allocation can reuse it without another lower-level allocation.

The following snapshots record:

- `allocated`: storage occupied by live tensors;
- `reserved`: the total memory currently managed by PyTorch's caching allocator;
- `free_cuda`: free memory reported by the CUDA runtime;
- `total_cuda`: the CUDA-visible memory capacity reported to this process.

The test allocation is bounded to 512 MiB. It uses `torch.empty`, so the experiment concerns allocation bookkeeping rather than initialization bandwidth.

In [3]:
import gc

import pandas as pd


MIB = 1024**2
GIB = 1024**3
TEST_ALLOCATION_MIB = 512
TEST_ALLOCATION_BYTES = TEST_ALLOCATION_MIB * MIB
DTYPE = torch.float32

element_size_bytes = torch.empty(
    (),
    dtype=DTYPE,
).element_size()

if TEST_ALLOCATION_BYTES % element_size_bytes != 0:
    raise ValueError("Test allocation must contain a whole number of elements")

element_count = TEST_ALLOCATION_BYTES // element_size_bytes

print(f"Test allocation: {TEST_ALLOCATION_MIB} MiB")
print(f"Data type: {DTYPE}")
print(f"Element size: {element_size_bytes} bytes")
print(f"Element count: {element_count:,}")

Test allocation: 512 MiB
Data type: torch.float32
Element size: 4 bytes
Element count: 134,217,728


In [ ]:
def cuda_memory_snapshot(stage: str) -> dict[str, int | float | str]:
    free_bytes, total_bytes = torch.cuda.mem_get_info(device)

    allocated_bytes = torch.cuda.memory_allocated(device)
    reserved_bytes = torch.cuda.memory_reserved(device)

    return {
        "stage": stage,
        "allocated_bytes": allocated_bytes,
        "reserved_bytes": reserved_bytes,
        "free_cuda_bytes": free_bytes,
        "total_cuda_bytes": total_bytes,
        "allocated_mib": allocated_bytes / MIB,
        "reserved_mib": reserved_bytes / MIB,
        "free_cuda_mib": free_bytes / MIB,
        "allocator_slack_mib": (reserved_bytes - allocated_bytes) / MIB,
    }


gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)
torch.cuda.synchronize(device)

memory_snapshots = [
    cuda_memory_snapshot("baseline after empty_cache"),
]

test_values = torch.empty(
    element_count,
    dtype=DTYPE,
    device=device,
)

torch.cuda.synchronize(device)
memory_snapshots.append(
    cuda_memory_snapshot("512 MiB tensor live"),
)

actual_tensor_bytes = test_values.numel() * test_values.element_size()

del test_values
gc.collect()
torch.cuda.synchronize(device)

memory_snapshots.append(
    cuda_memory_snapshot("tensor deleted; cache retained"),
)

torch.cuda.empty_cache()
torch.cuda.synchronize(device)

memory_snapshots.append(
    cuda_memory_snapshot("after final empty_cache"),
)

memory_frame = pd.DataFrame(memory_snapshots)

display(
    memory_frame[
        [
            "stage",
            "allocated_mib",
            "reserved_mib",
            "allocator_slack_mib",
            "free_cuda_mib",
        ]
    ].round(2)
)

print(f"Requested tensor storage: {TEST_ALLOCATION_BYTES / MIB:.2f} MiB")
print(f"Actual tensor storage: {actual_tensor_bytes / MIB:.2f} MiB")
print(f"Peak allocated memory: {torch.cuda.max_memory_allocated(device) / MIB:.2f} MiB")
print(f"Peak reserved memory: {torch.cuda.max_memory_reserved(device) / MIB:.2f} MiB")

,stage,allocated_mib,reserved_mib,allocator_slack_mib,free_cuda_mib
0,baseline after empty_cache,0.0,0.0,0.0,98941.73
1,512 MiB tensor live,512.0,512.0,0.0,98417.00
2,tensor deleted; cache retained,0.0,512.0,512.0,98419.47
3,after final empty_cache,0.0,0.0,0.0,98931.35


Requested tensor storage: 512.00 MiB
Actual tensor storage: 512.00 MiB
Peak allocated memory: 512.00 MiB
Peak reserved memory: 512.00 MiB


In [ ]:
baseline = memory_snapshots[0]
tensor_live = memory_snapshots[1]
tensor_deleted = memory_snapshots[2]
cache_cleared = memory_snapshots[3]

allocated_increase_bytes = tensor_live["allocated_bytes"] - baseline["allocated_bytes"]

assert allocated_increase_bytes >= actual_tensor_bytes
assert tensor_deleted["allocated_bytes"] < tensor_live["allocated_bytes"]
assert cache_cleared["reserved_bytes"] <= tensor_deleted["reserved_bytes"]

allocator_summary = pd.Series(
    {
        "tensor_storage_mib": actual_tensor_bytes / MIB,
        "observed_allocated_increase_mib": allocated_increase_bytes / MIB,
        "reserved_while_live_mib": tensor_live["reserved_mib"],
        "reserved_after_delete_mib": tensor_deleted["reserved_mib"],
        "reserved_after_empty_cache_mib": cache_cleared["reserved_mib"],
        "free_cuda_recovered_after_empty_cache_mib": (
            cache_cleared["free_cuda_bytes"] - tensor_deleted["free_cuda_bytes"]
        )
        / MIB,
    },
    name="value",
)

display(allocator_summary.to_frame().round(2))

,value
tensor_storage_mib,512.00
observed_allocated_increase_mib,512.00
reserved_while_live_mib,512.00
reserved_after_delete_mib,512.00
reserved_after_empty_cache_mib,0.00
free_cuda_recovered_after_empty_cache_mib,511.88


## Observations

TODO: Record allocation, transfer, and kernel measurements from actual runs.

## Explanation

TODO: Attribute costs to allocation, synchronization, transfer, or device access.

## Connection to LLMs

Weights, activations, and KV caches place sustained pressure on memory capacity and bandwidth.

## Further Exploration

TODO: Compare access patterns, data types, and reuse while controlling synchronization.